# 02 · Ablation Study — B0 → B1 → M1 → M2 → M3 → M4 → M5

This notebook runs the full ablation from `RESEARCH.md` chapters 5 and 8 on the **NVDA minute-bar panel**.

| Version | Description | Layer added |
| :--- | :--- | :--- |
| **B0** | Raw trend crossover, no guardrails | — |
| **B1** | B0 + execution guardrails (cooldown, no late-day entry, forced flatten) | Execution |
| **M1** | B1 + alpha confirmation (VWAP dev, volume surprise, proxy alignment) | Alpha |
| **M2** | M1 + rule-based regime gate | Regime |
| **M3** | M2 + volatility-scaled stop-loss | Risk |
| **M4** | M3 + volatility-targeted position sizing | Risk |
| **M5** | M4 + meta-label filter (triple-barrier classifier) | Alpha + ML |

Each version is a one-line change relative to the previous one, so we can attribute every bp of Sharpe improvement to a specific design decision.

In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT / 'src'))

from backtest import (
    BacktestConfig, run_backtest, compute_metrics,
    build_baseline_signal, build_alpha_confirmed_signal,
    apply_regime_gate, triple_barrier_labels, pretty_metrics_row,
)

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110

panel = pd.read_parquet(PROJECT / 'data_qtc' / 'combined' / 'research_panel.parquet')
nvda = panel[panel['symbol'] == 'NVDA'].sort_values('timestamp').reset_index(drop=True)
print(f'NVDA bars: {len(nvda):,}')
print(f'Date range: {nvda["timestamp"].min()}  ->  {nvda["timestamp"].max()}')

In [ ]:
results = {}
equities = {}
trade_logs = {}

def record(name, bars, trades):
    m = compute_metrics(bars, trades)
    results[name] = m
    equities[name] = bars[['timestamp', 'equity_ret']].copy()
    trade_logs[name] = trades
    row = pretty_metrics_row(name, m)
    return pd.DataFrame([row]).set_index('version')

## B0 — Raw trend crossover

`long_signal_baseline / short_signal_baseline` with zero guardrails, 1 bp slippage, $10k fixed notional. Expected to **bleed from turnover** — this is the null hypothesis.

In [ ]:
sig_b0 = build_baseline_signal(nvda)
cfg_b0 = BacktestConfig(
    cooldown_bars=0, flat_before_close_min=0, forced_flatten=False,
    stop_loss_k=None, max_hold_bars=None,
    slippage_bps=1.0, commission_bps=0.0,
    notional=10_000, use_vol_targeting=False,
)
bars, trades = run_backtest(nvda, sig_b0, cfg_b0)
record('B0', bars, trades)

## B1 — + Execution guardrails

Only change: add `cooldown_bars=5`, `flat_before_close_min=15`, `forced_flatten=True`. Same signal.

In [ ]:
cfg_b1 = BacktestConfig(
    cooldown_bars=5, flat_before_close_min=15, forced_flatten=True,
    stop_loss_k=None, max_hold_bars=None,
    slippage_bps=1.0, commission_bps=0.0,
    notional=10_000, use_vol_targeting=False,
)
bars, trades = run_backtest(nvda, sig_b0, cfg_b1)
record('B1', bars, trades)

## M1 — + Alpha confirmation

Require three things together to enter:
1. Baseline trend says long/short
2. `vwap_dev` same sign as trend and > 5 bps
3. `volume_surprise` ≥ 1.0 (not a dead tape)
4. At least one proxy (SPY / QQQ / XLK) agrees with direction

In [ ]:
sig_m1 = build_alpha_confirmed_signal(
    nvda, vwap_dev_k=0.0005, vol_surprise_min=1.0, require_proxy_align=True
)
print(f'M1 long : {int((sig_m1 == 1).sum()):,} bars  '
      f'({(sig_m1 == 1).mean():.1%})')
print(f'M1 short: {int((sig_m1 == -1).sum()):,} bars  '
      f'({(sig_m1 == -1).mean():.1%})')

bars, trades = run_backtest(nvda, sig_m1, cfg_b1)
record('M1', bars, trades)

## M2 — + Rule-based regime gate

Block entries on bars where we're in **high-vol + not-strong-trend**. This uses the precomputed flags `is_high_vol` and `is_strong_trend`.

In [ ]:
sig_m2 = apply_regime_gate(sig_m1, nvda)
print(f'M2 long : {int((sig_m2 == 1).sum()):,} bars')
print(f'M2 short: {int((sig_m2 == -1).sum()):,} bars')
print(f'Gate filters out {((sig_m1 != 0) & (sig_m2 == 0)).sum():,} of {(sig_m1 != 0).sum():,} M1 signals '
      f'({((sig_m1 != 0) & (sig_m2 == 0)).sum() / max(1, (sig_m1 != 0).sum()):.1%})')

bars, trades = run_backtest(nvda, sig_m2, cfg_b1)
record('M2', bars, trades)

## M3 — + Volatility-scaled stop loss

`stop_loss_k=2.0` means stop = 2 × `rolling_vol` × entry_price. Tighter stops in calm markets, wider stops in turbulent markets. Also cap holding time at 60 minutes so we don't rot in losing trades.

In [ ]:
cfg_m3 = BacktestConfig(
    cooldown_bars=5, flat_before_close_min=15, forced_flatten=True,
    stop_loss_k=2.0, max_hold_bars=60, trailing=False,
    slippage_bps=1.0, commission_bps=0.0,
    notional=10_000, use_vol_targeting=False,
)
bars, trades = run_backtest(nvda, sig_m2, cfg_m3)
record('M3', bars, trades)

## M4 — + Vol-targeted sizing

Scale notional inversely with volatility so every trade risks the same fraction of equity.

In [ ]:
cfg_m4 = BacktestConfig(
    cooldown_bars=5, flat_before_close_min=15, forced_flatten=True,
    stop_loss_k=2.0, max_hold_bars=60, trailing=False,
    slippage_bps=1.0, commission_bps=0.0,
    notional=10_000, use_vol_targeting=True, vol_target_daily=0.02,
)
bars, trades = run_backtest(nvda, sig_m2, cfg_m4)
record('M4', bars, trades)

## M5 — + Meta-label filter (triple-barrier + Logistic Regression)

Lopez de Prado *Advances in Financial Machine Learning*, Ch. 3 — train a classifier to predict whether a given primary signal will hit the profit barrier before the stop barrier. Only take trades the model believes in.

1. Label every bar with a primary signal using triple-barrier outcomes (`+1 / -1 / 0`)
2. For long signals, fit `P(label == +1)` using pre-trade features
3. Only act on signals where `P > 0.55`
4. Same logic for short side

We use a **time-based train/test split** (first 70% = train, last 30% = test). Only test-set trades are evaluated.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

meta_feature_cols = [
    'trend_strength', 'fast_minus_slow', 'price_minus_fast',
    'vwap_dev', 'volume_surprise', 'rolling_vol', 'abs_ret_5m',
    'hl_range', 'SPY_trend_strength', 'QQQ_trend_strength',
]

labels = triple_barrier_labels(nvda, upper_k=2.0, lower_k=1.5, max_hold_bars=30)
print('triple-barrier label distribution:')
print(labels.value_counts().to_dict())

split_i = int(0.7 * len(nvda))
df_full = nvda.copy()
df_full['label'] = labels.values
df_full['sig_m2'] = sig_m2.values

X_all = df_full[meta_feature_cols].replace([np.inf, -np.inf], np.nan)
valid = X_all.notna().all(axis=1) & df_full['label'].ne(0) & df_full['sig_m2'].ne(0)

# Long-side model
long_mask_train = valid & (df_full.index < split_i) & (df_full['sig_m2'] == 1)
long_mask_test  = valid & (df_full.index >= split_i) & (df_full['sig_m2'] == 1)

short_mask_train = valid & (df_full.index < split_i) & (df_full['sig_m2'] == -1)
short_mask_test  = valid & (df_full.index >= split_i) & (df_full['sig_m2'] == -1)

def fit_meta(X_train, y_train, pos_label):
    scaler = StandardScaler().fit(X_train)
    y_bin = (y_train == pos_label).astype(int)
    if y_bin.sum() < 50 or y_bin.sum() > len(y_bin) - 50:
        return None, None
    clf = LogisticRegression(class_weight='balanced', max_iter=500, C=0.5)
    clf.fit(scaler.transform(X_train), y_bin)
    return clf, scaler

X_l_train = X_all.loc[long_mask_train].values
y_l_train = df_full.loc[long_mask_train, 'label'].values
clf_long, sc_long = fit_meta(X_l_train, y_l_train, pos_label=1)

X_s_train = X_all.loc[short_mask_train].values
y_s_train = df_full.loc[short_mask_train, 'label'].values
clf_short, sc_short = fit_meta(X_s_train, y_s_train, pos_label=-1)

print(f'Long-side model trained on {long_mask_train.sum():,} rows')
print(f'Short-side model trained on {short_mask_train.sum():,} rows')

In [ ]:
sig_m5 = pd.Series(0, index=nvda.index, dtype=int)
threshold = 0.55

if clf_long is not None:
    X_l_test = X_all.loc[long_mask_test].values
    p_long = clf_long.predict_proba(sc_long.transform(X_l_test))[:, 1]
    accept = p_long >= threshold
    idx_accept = long_mask_test[long_mask_test].index[accept]
    sig_m5.loc[idx_accept] = 1

if clf_short is not None:
    X_s_test = X_all.loc[short_mask_test].values
    p_short = clf_short.predict_proba(sc_short.transform(X_s_test))[:, 1]
    accept = p_short >= threshold
    idx_accept = short_mask_test[short_mask_test].index[accept]
    sig_m5.loc[idx_accept] = -1

print(f'M5 long : {int((sig_m5 == 1).sum()):,} bars  (M2 long was {int((sig_m2 == 1).sum()):,})')
print(f'M5 short: {int((sig_m5 == -1).sum()):,} bars  (M2 short was {int((sig_m2 == -1).sum()):,})')

# Evaluate only on the test window for fairness
nvda_test = nvda.iloc[split_i:].reset_index(drop=True).copy()
sig_m5_test = sig_m5.iloc[split_i:].reset_index(drop=True)
sig_m4_test = sig_m2.iloc[split_i:].reset_index(drop=True)

# For apples-to-apples, also re-run M4 on the test window
bars_m4_oos, trades_m4_oos = run_backtest(nvda_test, sig_m4_test, cfg_m4)
record('M4_OOS', bars_m4_oos, trades_m4_oos)

bars, trades = run_backtest(nvda_test, sig_m5_test, cfg_m4)
record('M5', bars, trades)

## Ablation summary

In [ ]:
order = ['B0', 'B1', 'M1', 'M2', 'M3', 'M4', 'M4_OOS', 'M5']
rows = [pretty_metrics_row(n, results[n]) for n in order if n in results]
summary_df = pd.DataFrame(rows).set_index('version')
summary_df

In [ ]:
raw_rows = []
for n in order:
    if n not in results: continue
    m = results[n]
    raw_rows.append({
        'version': n,
        'ann_return': m.get('ann_return', np.nan),
        'sharpe': m.get('sharpe', np.nan),
        'sortino': m.get('sortino', np.nan),
        'max_dd': m.get('max_drawdown', np.nan),
        'n_trades': m.get('n_trades', 0),
        'win_rate': m.get('win_rate', np.nan),
    })
numeric = pd.DataFrame(raw_rows).set_index('version')

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
numeric['sharpe'].plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].axhline(0, color='k', lw=0.8)
axes[0].set_title('Sharpe by version')

numeric['ann_return'].plot(kind='bar', ax=axes[1], color='seagreen')
axes[1].axhline(0, color='k', lw=0.8)
axes[1].set_title('Annualised return by version')
axes[1].yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))

numeric['max_dd'].plot(kind='bar', ax=axes[2], color='indianred')
axes[2].set_title('Max drawdown by version')
axes[2].yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
for ax in axes: ax.tick_params(axis='x', rotation=0)
fig.tight_layout()

## Equity curves overlaid

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
palette = sns.color_palette('tab10', n_colors=len(order))
for color, name in zip(palette, order):
    if name not in equities: continue
    eq = equities[name]
    ax.plot(pd.to_datetime(eq['timestamp']), eq['equity_ret'].values,
            label=f'{name}  (Sharpe {results[name].get("sharpe", 0):.2f})',
            lw=1.1, color=color)
ax.axhline(0, color='k', lw=0.7)
ax.set_title('Cumulative return (NVDA, 1-minute bars)')
ax.set_ylabel('cumulative return (sum of per-trade returns)')
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
ax.legend(loc='best', fontsize=9)
fig.tight_layout()

## Per-trade return distribution

A good ablation shouldn't just improve mean PnL — it should **reshape the trade distribution**: cut the losing tail, keep the winning tail. The boxplots below make that visible.

In [ ]:
chunks = []
for n in order:
    if n not in trade_logs: continue
    t = trade_logs[n]
    if t.empty: continue
    chunks.append(pd.DataFrame({'version': n, 'ret': t['ret'].values}))
all_trades = pd.concat(chunks, ignore_index=True)

fig, ax = plt.subplots(figsize=(12, 4))
sns.boxplot(data=all_trades, x='version', y='ret', ax=ax, fliersize=2,
            order=[n for n in order if n in trade_logs and not trade_logs[n].empty])
ax.axhline(0, color='k', lw=0.8)
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
ax.set_title('Per-trade return distribution by version (outliers clipped at 1.5×IQR)')
fig.tight_layout()

## Exit-reason breakdown

What *caused* each trade to close? If stops dominate, the stop is probably too tight. If `flip` dominates, the signal is thrashing. If `eod_flat` dominates, we're being saved by the session-end rule rather than by the strategy.

In [ ]:
exit_rows = []
for n in order:
    if n not in trade_logs or trade_logs[n].empty: continue
    counts = trade_logs[n]['exit_reason'].value_counts(normalize=True)
    for r, v in counts.items():
        exit_rows.append({'version': n, 'reason': r, 'share': v})
exit_df = pd.DataFrame(exit_rows)
if not exit_df.empty:
    pivot = exit_df.pivot(index='version', columns='reason', values='share').fillna(0)
    pivot = pivot.reindex([n for n in order if n in pivot.index])
    pivot.plot(kind='bar', stacked=True, figsize=(10, 3.5), colormap='tab20')
    plt.title('Exit reasons (share of trades)')
    plt.ylabel('share')
    plt.xticks(rotation=0)
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
    plt.tight_layout()

## Save outputs

In [ ]:
outdir = PROJECT / 'results'
outdir.mkdir(exist_ok=True)
summary_df.to_csv(outdir / 'ablation_summary.csv')
for n, df in trade_logs.items():
    if df is None or df.empty: continue
    df.to_parquet(outdir / f'trades_{n}.parquet', index=False)
for n, eq in equities.items():
    eq.to_parquet(outdir / f'equity_{n}.parquet', index=False)
print('Saved to', outdir)

## How to read these numbers

- **B0 vs B1**: the *only* difference is execution rules. The gap between them is how much a naive backtest overstates what you can actually harvest once you respect cooldown / session rules / no pyramiding.
- **B1 → M1 → M2**: these layers trade *fewer* times but each trade should be higher quality (win rate up, winner/loser ratio up, right-tail retention up). If turnover drops 5× but return only drops 2×, the layer is net-positive.
- **M2 → M3**: introducing the stop should cap `max_dd` and change `exit_reason` distribution toward `stop`. If `stop_frequency` is > 30%, the stop is too tight.
- **M3 → M4**: vol targeting should **not** change Sharpe much in isolation; it stabilises daily PnL and is the precondition for combining multiple symbols in future work.
- **M4 → M5**: meta-label is the out-of-sample honesty check. If M5 OOS Sharpe > M4 OOS Sharpe, the classifier found genuine signal. If M5 OOS ≈ M4 OOS, the primary signal is already as clean as the features allow.

All seven versions run on the same 125K-row NVDA panel with the same cost assumptions, so the differences here are attributable to the design decisions, not to configuration noise. This is the payload for the `RESEARCH.md` ablation section.